In [ ]:
# Cell 0: API keys / credentials
# Open-Meteo's historical weather API does not require an API key.
# Placeholder kept for consistency with other notebooks in this toolkit.
OPEN_METEO_API_KEY = None

# 03 — HDD/CDD Calculation & Weather-Normalized Demand

Computes population-weighted Heating/Cooling Degree Days (base 18°C)
per country from Open-Meteo historical temperature data, joins them to
the gas flow data ingested in notebook 01, and compares **observed**
demand against **weather-normalized** demand (i.e. what demand would
have been under average weather conditions for the sample period).

Weather normalization here uses a simple OLS regression of demand on
HDD and CDD per country, then removes the weather-driven deviation
from each day's HDD/CDD relative to the sample mean. This is a
standard degree-day normalization approach; it does not correct for
other demand drivers (economic activity, storage dynamics, price
response), which would need a richer model.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().parent / "data"))

from entsog_client import COUNTRIES, EntsogClient, resolve_analysis_date
from weather_client import WeatherClient, calculate_hdd_cdd, HDD_CDD_BASE_TEMP_C

In [ ]:
# ANALYSIS_DATE = None auto-resolves to the latest available ENTSOG data point.
# Set an explicit date(YYYY, M, D) to pin the run to a specific end date instead.
ANALYSIS_DATE = None

data_dir = Path.cwd().parent / "data"

resolved_date = resolve_analysis_date(ANALYSIS_DATE)
print(f"ANALYSIS_DATE resolved to: {resolved_date}")
print(f"HDD/CDD base temperature: {HDD_CDD_BASE_TEMP_C}\u00b0C")

## Load ingested gas flow data

Reads the most recent `entsog_flows_*.parquet` written by notebook 01.
Run notebook 01 first if this file doesn't exist yet.

In [ ]:
flow_files = sorted(data_dir.glob("entsog_flows_*.parquet"))
if not flow_files:
    raise FileNotFoundError(
        "No entsog_flows_*.parquet found in data/ - run notebook 01 first."
    )

flows_df = pd.read_parquet(flow_files[-1])

# Daily national demand proxy: sum of all reported physical flow values
# per country per day. A production-grade demand measure would filter to
# specific point/direction classifications (e.g. exit points only); this
# notebook uses the simpler total-flow sum as a stand-in.
demand_df = (
    flows_df.groupby(["queryCountry", "period"], as_index=False)["value"]
    .sum()
    .rename(columns={"queryCountry": "country_code", "value": "demand"})
)
demand_df["period"] = pd.to_datetime(demand_df["period"])
demand_df.head()

## Fetch population-weighted temperature & compute HDD/CDD

In [ ]:
weather_client = WeatherClient()

start_date = demand_df["period"].min().date()
end_date = demand_df["period"].max().date()

hdd_cdd_frames = []
for country in COUNTRIES:
    temp_df = weather_client.get_country_temperature(country, start_date, end_date)
    if temp_df.empty:
        continue
    hdd_cdd_frames.append(calculate_hdd_cdd(temp_df))

hdd_cdd_df = pd.concat(hdd_cdd_frames, ignore_index=True)
hdd_cdd_df["date"] = pd.to_datetime(hdd_cdd_df["date"])
hdd_cdd_df.head()

## Join demand with HDD/CDD

In [ ]:
merged_df = demand_df.merge(
    hdd_cdd_df,
    left_on=["country_code", "period"],
    right_on=["country_code", "date"],
    how="inner",
).drop(columns="date")
merged_df.head()

## Weather normalization

For each country, fit `demand ~ HDD + CDD` by OLS to estimate weather
sensitivity (mcm per degree-day), then compute weather-normalized
demand by removing the modeled effect of each day's HDD/CDD deviating
from the sample-period mean:

```
normalized_demand = observed_demand
                     - beta_hdd * (hdd - mean(hdd))
                     - beta_cdd * (cdd - mean(cdd))
```

In [ ]:
def weather_normalize(group: pd.DataFrame) -> pd.DataFrame:
    X = np.column_stack([np.ones(len(group)), group["hdd"], group["cdd"]])
    y = group["demand"].to_numpy()
    coeffs, *_ = np.linalg.lstsq(X, y, rcond=None)
    intercept, beta_hdd, beta_cdd = coeffs

    mean_hdd = group["hdd"].mean()
    mean_cdd = group["cdd"].mean()

    group = group.copy()
    group["weather_normalized_demand"] = group["demand"] - beta_hdd * (
        group["hdd"] - mean_hdd
    ) - beta_cdd * (group["cdd"] - mean_cdd)
    group["beta_hdd"] = beta_hdd
    group["beta_cdd"] = beta_cdd
    return group


normalized_df = pd.concat(
    [weather_normalize(group) for _, group in merged_df.groupby("country_code")],
    ignore_index=True,
)
normalized_df[["country_code", "period", "demand", "hdd", "cdd", "weather_normalized_demand"]].head()

## Observed vs weather-normalized demand

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(COUNTRIES), 1, figsize=(10, 3 * len(COUNTRIES)), sharex=True)
for ax, country in zip(axes, COUNTRIES):
    country_df = normalized_df[normalized_df["country_code"] == country].sort_values("period")
    ax.plot(country_df["period"], country_df["demand"], label="Observed", alpha=0.7)
    ax.plot(country_df["period"], country_df["weather_normalized_demand"], label="Weather-normalized", alpha=0.7)
    ax.set_title(country)
    ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
output_path = data_dir / f"hdd_cdd_demand_normalized_{resolved_date.isoformat()}.parquet"
normalized_df.to_parquet(output_path, index=False)
print(f"Wrote {len(normalized_df)} rows to {output_path}")